In [1]:
data_folder="../Data/" #and libraries and funcitons
import sys
import tensorflow as tf
import numpy as np
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import tqdm


sys.path.append(os.path.abspath("../Codes"))

sys.path.append(os.path.abspath("../_Deep_Learning"))

from Utils import split_create_windows, scale_newax, compute_psd
from Utils import freq_filter, windowize,  RegionWiseStandardizer

from BCI_Library import read_subject, saveresults_pickle, compute_welch, extract_features_from_spectra_more_bands
from BCI_Library import  train_single_model,select_regions_Cohen_effect_size, build_compiled_mlp


def compute_power_spectra(data_2_MI, data_2_Rest, starttime=250, endtime=250):
    sfreq = 250
    miff = [8,12]
    maff = [12,30]

    WMI, WRe, FreqMI = compute_welch(data_2_MI, data_2_Rest, sfreq,
                                        kargs_welch={"fmin":miff[0], "fmax":maff[-1]},
                                        starttime=starttime, endtime=endtime,)

    return WMI, WRe


def selezione_cohen(WRe, WMI, train_idx, num_best_ROIs):
    training_index_Re = train_idx[np.where(train_idx<len(WRe))] 
    training_index_MI = train_idx[np.where(train_idx>=len(WRe))]-len(WRe)

    regions_selected, effect_size  = select_regions_Cohen_effect_size(wpsdMI=WMI[training_index_MI], 
                                                    wpsdRest=WRe[training_index_Re], 
                                                        important_regions=[], nbest_regions=num_best_ROIs)
    return regions_selected

2026-05-27 09:42:37.305102: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-27 09:42:37.383218: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-27 09:42:39.119548: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# Investigate performaces per time
#del train_single_model_investigate2
# To accept the input in the form (X_train, X_val..) and not (X, train_idx, val_idx)
def train_single_model_investigate2(X_train, X_val, y_train, y_val, method, 
                scaler=StandardScaler, seed=224, return_predictions=True,
                verbose=False, fit_kwargs={}, n_windows=1):
    
    # StandardScaler:
    if scaler is not None:
        scaler_instance = scaler()
        X_train_scaled = scaler_instance.fit_transform(X_train)
        X_val_scaled   = scaler_instance.transform(X_val)

    # Model instance
    try:
        model = method(random_state=seed)
    except:
        try:
            model = method()
        except:
            model = method
    is_keras_model = isinstance(model, tf.keras.Model)

    if is_keras_model:
        model.fit(X_train_scaled, y_train,validation_data=(X_val_scaled, y_val), **fit_kwargs)
    else:
        model.fit(X_train_scaled, y_train)
    acc = {}
    confusion_mat = {}
    labels_preds = {}
    for _ in range(n_windows):
        y_pred = model.predict(X_val_scaled[_::n_windows])
        labels_preds["win_"+str(_)] = {"y_true":y_val[_::n_windows], "y_pred":y_pred}
    
        if is_keras_model:
            y_pred = (y_pred > 0.5).astype(int).ravel()

        # Accuracy   # Confusion matrix
        acc["win_"+str(_)] = accuracy_score(y_val[_::n_windows], y_pred)
        confusion_mat["win_"+str(_)] = confusion_matrix(y_val[_::n_windows], y_pred)
        #print(len(y_val[_::n_windows]))
    if return_predictions:
        return acc, confusion_mat, model, labels_preds
    return acc, confusion_mat, model

# To accept the input in the form (MIandRest) and not (MI,Rest) 
def extract_features_from_spectra_more_bands_2(wpsd, frequiMI,
                    min_freq_frature=[4,8,12], max_freq_frature=[8,12,30], 
                    select_mean=True, select_max=True,
                    reshape_features=True):
    """
    Extract frequency-domain features from MI and Rest spectra of data.
    Features are the mean and max in a fixed window of freqs (min_freq_frature, max_freq_frature)
    For the selected ROIs. Selected previously
    ----------
    
    - wpsdxx -> welch spectra of the data: (n_trials x n_ROIs x n_freqs) # Works also with other way of estimating Power, not necessarly Welch (!?)
    - frequencies corresponding to the bins in welch spectra (n_freqs)

    Returns
    -------
    - features: mean and max of each selected ROI, 
        computed in the freq interval (min_freq_frature,max_freq_frature)

    """
    if isinstance(min_freq_frature, int) or isinstance(min_freq_frature, float):
        min_freq_frature = [min_freq_frature]
    if isinstance(max_freq_frature, int) or isinstance(max_freq_frature, float):
        max_freq_frature = [max_freq_frature]
        
    # selecting the freq range on wich compute features 
    # as features i chose the max and the maen of selected_ROIs 
    # in freqs (min_freq_frature, min_freq_frature)

    ######################## Extract the features ########################

    start_freq_idx = [np.where(frequiMI>=min_freq_frature[i])[0][0] for i in range(len(min_freq_frature))]
    end_freq_idx = [np.where(frequiMI<=max_freq_frature[i])[0][-1] for i in range(len(max_freq_frature))]
    features = []
    
    for _ in range(len(min_freq_frature)):

        wpsd_freqs = wpsd[:,start_freq_idx[_]:end_freq_idx[_]+1]
      

        if select_mean:
            features.append(wpsd_freqs.mean(axis=-1,keepdims=True))

        if select_max:  
            features.append(wpsd_freqs.max(axis=-1,keepdims=True))
    
    features = np.concatenate(features,axis=-1)

    # features.shape (Trials, selected_ROIs, len(min_freq_feature*2))
    
    if reshape_features:
        features = features.reshape(features.shape[0],-1)

    return features

## Train on feedback (3 - 6 ) sec<br> Test on all (1 - 6 ) sec

### Cohen's d effect size - EEG concat MEG - region selection starting at 3s

Selection consistent with DL:
- Select 8 Regions of EEG;
- Select 8 Regions of MEG;
- Unione dei set EEG U MEG;
- Take both EEG and MEG of Regions EEG U MEG
 "../Results/BCI_Performances/All_subjects_BCI_Performances_ultimate_region_selection_from_3s_1s_windows.pkl

In [ ]:
#Integrate EEG EMG
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm

start_time_selection_regions = 750 # 3sec
seed = 224
random_seed = 224
DataType = "EEG concat MEG"
# keys of the performance dictionary

miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "PowerSpectra - MeanMax band"
Region_Selection= "Cohen's d effect size selection" # "-"
Comments = "Windows of 1sec, Train on sec 3-6, test on sec 1-6"
PFR = False

num_best_ROIs = 8 # how manu ROIs to extract for modality (then Faccio unione tra regioni EEG e MEG)

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

verbose = False

input_shape = 256   # length of each window
shift = 85          # points to shift for next window in data

start_val_test = 1           # seconds where to start to extract windows
sampling_hz = 250;  start_val_test = start_val_test*sampling_hz
num_windows_val_test = 12    # how many windows to extract from each trial
end_val_test = start_val_test + (num_windows_val_test-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)

start_train = 3           # seconds where to start to extract windows
start_train = start_train*sampling_hz
num_windows_train = 7     # how many windows to extract from each trial
end_train = start_train + (num_windows_train-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)

log_spectra = False             # if compute log of spectra after
fmin_spectra = 7                # min freq considered in spectra (output)
fmax_spectra = 31               # max freq considered in spectra (output)
bandwidth_spectra = 6           # smoothing for computing spectra (6 seems a good compromise)

Classifier_names, meths = ["SVC"], [SVC]  # ["MLP"], ["mlp"] # ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}


for Classifier_name, meth in zip(Classifier_names, meths):

    Rows = pd.DataFrame({
        "ROIs": [],
        "NROIs":[],
        "DataType": [],
        "freqs_band": [],  
        "subject": [],
        "Classifier": [],
        "Accuracy": [],
        "Features": [],
        "Comments": [],
        "Region Selection": [],
        "Date": []
    })
    print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
    
    
    



    for subject_index in tqdm.tqdm(range(20)):
        tutti_fold_selezioni_regioni = []
        fold_accuracies = []
        all_predictions = []
        confusions_matrices = []
        
        data_2_Rest_EEG = read_subject(data_folder, "EEG", "Baseline",subject_index,verbose=verbose)
        data_2_MI_EEG = read_subject(data_folder, "EEG", "MI",subject_index,verbose=verbose)
        data_EEG = np.concatenate((data_2_Rest_EEG, data_2_MI_EEG), axis=0)
        WMI_EEG, WRe_EEG = compute_power_spectra(data_2_MI_EEG,data_2_Rest_EEG,starttime=start_time_selection_regions)
        data_EEG = freq_filter(data_EEG, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
        
        data_2_Rest_MEG = read_subject(data_folder, "MEG", "Baseline",subject_index,verbose=verbose)
        data_2_MI_MEG = read_subject(data_folder, "MEG", "MI",subject_index,verbose=verbose)
        data_MEG = np.concatenate((data_2_Rest_MEG, data_2_MI_MEG), axis=0)
        WMI_MEG, WRe_MEG = compute_power_spectra(data_2_MI_MEG,data_2_Rest_MEG, starttime=start_time_selection_regions)
        data_MEG = freq_filter(data_MEG, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
        

        # y shape: (n_trials,) with negative values for Rest and positive for MI
        y_EEG = np.concatenate([-np.ones((data_2_Rest_EEG.shape[0])), np.ones((data_2_MI_EEG.shape[0]))])
        y_MEG = np.concatenate([-np.ones((data_2_Rest_MEG.shape[0])), np.ones((data_2_MI_MEG.shape[0]))])

        assert np.array_equal(y_EEG, y_MEG), "Le etichette di EEG e MEG non corrispondono. Controlla i dati."


        # Data_EEG, WMI_EEG, labels_EEG, Feat_extraction_EEG, Feat_MI_EEG
       

        

        for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data_EEG, y_EEG)):
            Feat_train_regions_selected = None
            Feat_test_regions_selected = None

            # selecting region based on training data only
            regions_selected_EEG = selezione_cohen(WRe_EEG, WMI_EEG, train_idx, num_best_ROIs)
            regions_selected_MEG = selezione_cohen(WRe_MEG, WMI_MEG, train_idx, num_best_ROIs)
            regions_selected = list(set(regions_selected_EEG) | set(regions_selected_MEG))
            
            regions_selected.sort()
            print("\n Per il fold ", fold_idx, " Ho selezionato un numero di regioni pari a ", len(regions_selected), regions_selected)
            tutti_fold_selezioni_regioni.append(regions_selected)


            for region_index, regione in enumerate(regions_selected):
        
                split_num = fold_idx+1
                
                x_train_MEG, y_train_MEG, x_val_MEG, y_val_MEG, x_test_MEG, y_test_MEG = (
                split_create_windows(data_MEG, y_MEG, train_idx, block_idx, 1, win_len=input_shape, shift=shift,
                    random_seed=random_seed, regione=regione,
                    start_train=start_train, end_train=end_train,
                    start_val_test=start_val_test, end_val_test=end_val_test)
                    )

                psd_train_MEG, psd_val_MEG, psd_test_MEG, freqs = compute_psd(
                x_train_MEG, x_val_MEG, x_test_MEG, y_train_MEG, y_val_MEG, y_test_MEG, 
                sampling_hz, fmin_spectra, fmax_spectra, bandwidth_spectra, log_spectra, scaler=None
                )
                
                x_train_EEG, y_train_EEG, x_val_EEG, y_val_EEG, x_test_EEG, y_test_EEG = (
                split_create_windows(data_EEG, y_EEG, train_idx, block_idx, 1, win_len=input_shape, shift=shift,
                    random_seed=random_seed, regione=regione,
                    start_train=start_train, end_train=end_train,
                    start_val_test=start_val_test, end_val_test=end_val_test)
                    )
                
                psd_train_EEG, psd_val_EEG, psd_test_EEG, freqs = compute_psd(
                x_train_EEG, x_val_EEG, x_test_EEG, y_train_EEG, y_val_EEG, y_test_EEG, 
                sampling_hz, fmin_spectra, fmax_spectra, bandwidth_spectra, log_spectra, scaler=None
                )
                
                assert np.array_equal(y_train_EEG, y_train_MEG), "Le etichette di train EEG e MEG non corrispondono. Controlla i dati."
                assert np.array_equal(y_val_EEG, y_val_MEG), "Le etichette di val EEG e MEG non corrispondono. Controlla i dati."
                assert np.array_equal(y_test_EEG, y_test_MEG), "Le etichette di test EEG e MEG non corrispondono. Controlla i dati."
            
        
                
                Features_training_1_regione_EEG = extract_features_from_spectra_more_bands_2(
                    psd_train_EEG, freqs,
                min_freq_frature=miff, max_freq_frature=maff, 
                select_mean=True, select_max=True,
                reshape_features=False)

                Features_test_1_regione_EEG = extract_features_from_spectra_more_bands_2(
                    psd_test_EEG, freqs,
                min_freq_frature=miff, max_freq_frature=maff, 
                select_mean=True, select_max=True,
                reshape_features=False)
            
                Features_training_1_regione_MEG = extract_features_from_spectra_more_bands_2(
                    psd_train_MEG, freqs,
                min_freq_frature=miff, max_freq_frature=maff, 
                select_mean=True, select_max=True,
                reshape_features=False)

                Features_test_1_regione_MEG = extract_features_from_spectra_more_bands_2(
                    psd_test_MEG, freqs,
                min_freq_frature=miff, max_freq_frature=maff, 
                select_mean=True, select_max=True,
                reshape_features=False)
                # Features_training_1_regione = model.encoder(x_train)
                # Features_test_1_regione = model.encoder(x_test)
                # Features_training_1_regione  shape (n_windows,16)
                Features_training_1_regione = np.concatenate((Features_training_1_regione_EEG, Features_training_1_regione_MEG), axis=-1)
                Features_test_1_regione = np.concatenate((Features_test_1_regione_EEG, Features_test_1_regione_MEG), axis=-1)

                
                    
                ################################### Feature Extractions ##########################################
                if Feat_train_regions_selected is None:
                    Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                    Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione), axis = 1) # asse di regione
                Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected, Features_test_1_regione), axis = 1) # asse di regione

            #####################################################################################################################


            # if Classifier_name.lower()=="mlp":
            #     meth = build_compiled_mlp(input_dim=Feat_All_regions_selected_concat.shape[1])
                
            acc, cm, model_classifier, predictions = train_single_model_investigate2(Feat_train_regions_selected, Feat_test_regions_selected,
                                            (y_train_EEG > 0).astype(int), (y_test_EEG > 0).astype(int), return_predictions=True,
                                            method=meth, seed=224, fit_kwargs=MLP_fit_kwargs,
                                            n_windows = num_windows_val_test)
            

            fold_accuracies.append(acc)
            all_predictions.append(predictions)
            confusions_matrices.append(cm)


            if verbose:
                print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                print(cm)

            #####################################################################################################################

        new_row = {
            "ROIs": [[regions_selected_EEG,regions_selected_MEG]],
            "NROIs":[len(regions_selected_EEG)],
            "DataType": [DataType],
            "freqs_band": [freqs_band],  
            "subject": [subject_index],
            "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
            "Features": [Features_name],
            "Comments": [Comments], 
            "Region Selection": [Region_Selection],
            "Date": [today],
            "Personalized_Freq_Range": [PFR],
        }

        # with open(f"../Results/BCI_Performances/Across_time_Power_Spectra_Confrontabili_with_DNN_windows_1s/Performances_across_time_subject_{subject_index}.pkl", "wb") as f:
        #     pickle.dump({
        #         "Details": new_row,
        #         "Selected_regions_per_fold_EEG_U_MEG": tutti_fold_selezioni_regioni,
        #         "all_predictions": all_predictions,
        #         "fold_accuracies": fold_accuracies
        #                 }, f)

### Investigate

In [ ]:
import pandas as pd
accmedie = []
for subject_index in range(20):
    subject_result_across_time = pd.read_pickle(f"../Results/BCI_Performances/Across_time_Power_Spectra_Confrontabili_with_DNN_windows_1s/Performances_across_time_subject_{subject_index}.pkl",)
    fold_accuracies = subject_result_across_time["fold_accuracies"]
    all_predictions = subject_result_across_time["all_predictions"]
    Details = subject_result_across_time["Details"]
    Selected_regions_per_fold_EEG_U_MEG = subject_result_across_time["Selected_regions_per_fold_EEG_U_MEG"]

    num_windows =len(fold_accuracies[0])


    # BALANCING of PREDICTIONs
    quanta_perc_1_per_fold = []
    for fold in range(5):
        somma = 0
        den = 0
        for win_dow in range(num_windows): 
            somma += all_predictions[fold][f"win_{win_dow}"]["y_pred"].sum()
            den += len(all_predictions[fold][f"win_{win_dow}"]["y_pred"])
        quanta_perc_1_per_fold.append(somma/den)
    quanta_perc_1_per_fold


    # PLOTS (per time)
    conteggio = 0; somma = 0
    accuracy_per_time = {}
    accuracy_per_fold = {}
    for key in fold_accuracies[0]:
        accuracy_per_time[key] = 0
    for _ in range(len(fold_accuracies)):
        for key in fold_accuracies[_]:
            conteggio += 1
            somma += fold_accuracies[_][key]
            accuracy_per_time[key]+= fold_accuracies[_][key]/5
            accuracy_per_fold[_] =  sum(list(fold_accuracies[_].values())[-7:]) / 7



    import numpy as np
    import matplotlib.pyplot as plt

    tag = "cohen_specific"


    # ---------- PRECOMPUTE COMMON AXIS ----------
    winds = sorted(all_predictions[0], key=lambda w: int(w.split("_")[1]))
    x = np.arange(len(winds))

    # ---------- CLASS-WISE ACCURACY (LEFT PANEL) ----------
    m0, s0, m1, s1 = [], [], [], []

    for w in winds:
        a0, a1 = [], []
        for fold in all_predictions:
            y = np.array(fold[w]["y_true"])
            p = np.array(fold[w]["y_pred"])
            a0.append((p[y == 0] == 0).mean())
            a1.append((p[y == 1] == 1).mean())
        m0.append(np.mean(a0)); s0.append(np.std(a0))
        m1.append(np.mean(a1)); s1.append(np.std(a1))

    m0, s0, m1, s1 = map(np.array, (m0, s0, m1, s1))

    # ---------- GLOBAL ACCURACY (RIGHT PANEL) ----------
    times = list(fold_accuracies[0])
    vals = np.array([[fold[t] for t in times] for fold in fold_accuracies])
    m, s = vals.mean(0), vals.std(0)

    # ---------- PLOTTING ----------
    fig, axs = plt.subplots(1, 2, figsize=(12, 5.5))

    # RIGHT
    ax = axs[1]
    ax.plot(x, m0, label="Rest")
    ax.fill_between(x, m0 - s0, m0 + s0, alpha=.2)

    ax.plot(x, m1, label="MI")
    ax.fill_between(x, m1 - s1, m1 + s1, alpha=.2)

    ax.set_title(f"Performances vs time - Class-wise")
    ax.set_xticks(x)
    ax.set_xticklabels(winds, rotation=45)
    ax.set_ylabel("accuracy")
    ax.legend()

    ymin, ymax = ax.get_ylim()
    ax.vlines(5, ymin, ymax, linestyles="dashed", colors="red", label="Start of feedback")
    ax.set_ylim(ymin, ymax)

    # LEFT
    ax = axs[0]

    ax.set_ylabel("accuracy")

    ax.plot(times, m)
    ax.scatter(times, list(accuracy_per_time.values()))
    ax.fill_between(times, m - s, m + s, alpha=.2)

    ax.set_xticks(x)
    ax.set_xticklabels(winds, rotation=45)
    ax.set_title(f"Performances vs time")

    ymin, ymax = ax.get_ylim()
    ax.vlines(5, ymin, ymax, linestyles="dashed", colors="red", label="Start of feedback")
    ax.set_ylim(ymin, ymax)
    plt.legend()
    plt.suptitle(f"Subject {subject_index} - {tag}\npercentage of predicted 1: {np.mean(quanta_perc_1_per_fold):.2f}\n"+
                 f"{np.mean(list(accuracy_per_fold.values())):.2f} Accuracy of DEEP on this subject")
    plt.tight_layout()
    plt.savefig(f"../Images/Performance_across_time/Perf_vs_Time_sbj{subject_index}_{tag}_test_start_from_1sec_train_from_3sec")
    accmedie.append(np.mean(list(accuracy_per_fold.values())))


## Train on feedback (3 - 6 ) sec<br> Test on feedback (3 - 6 ) sec

### Cohen's d effect size - EEG concat MEG - region selection starting at 3s

Selection consistent with DL:
- windows of 1s
- Select 8 Regions of EEG;
- Select 8 Regions of MEG;
- Unione dei set EEG U MEG;
- Take both EEG and MEG of Regions EEG U MEG

In [3]:
#Integrate EEG EMG
import pickle
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import date
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import StratifiedKFold
import tqdm

start_time_selection_regions = 750 # 3sec
seed = 224
random_seed = 224
DataType = "EEG concat MEG"
# keys of the performance dictionary

miff = [8,12]
maff = [12,30]
freqs_band = tuple([(i,j) for i,j in zip(miff,maff)])
Features_name = "PowerSpectra - MeanMax band"
Region_Selection= "Cohen's d effect size selection starting from 3s" # "-"
Comments = "Total regions = EEG_regions U MEG_regions"
PFR = False

num_best_ROIs = 8 # how manu ROIs to extract for modality (then Faccio unione tra regioni EEG e MEG)

n_folds=5
kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
today = date.today()
sfreq = 250

verbose = False

input_shape = 256   # length of each window
shift = 85          # points to shift for next window in data

start_val_test = 3           # seconds where to start to extract windows
sampling_hz = 250;  start_val_test = start_val_test*sampling_hz
num_windows_val_test = 7    # how many windows to extract from each trial
end_val_test = start_val_test + (num_windows_val_test-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)

start_train = 3           # seconds where to start to extract windows
start_train = start_train*sampling_hz
num_windows_train = 7     # how many windows to extract from each trial
end_train = start_train + (num_windows_train-1)*shift + input_shape  # seconds where to end to extract windows (1500 == 6 seconds)

log_spectra = False             # if compute log of spectra after
fmin_spectra = 7                # min freq considered in spectra (output)
fmax_spectra = 31               # max freq considered in spectra (output)
bandwidth_spectra = 6           # smoothing for computing spectra (6 seems a good compromise)

Classifier_names, meths = ["SVC"], [SVC]  # ["MLP"], ["mlp"] # ["LDA", "SVC", "RandomForest"],[LDA, SVC, RandomForestClassifier]
MLP_fit_kwargs={'epochs':20, 'batch_size':32}

Rows_all_subjects = pd.DataFrame({
    "ROIs": [],
    "NROIs":[],
    "DataType": [],
    "freqs_band": [],  
    "subject": [],
    "Classifier": [],
    "Accuracy": [],
    "Features": [],
    "Comments": [],
    "Region Selection": [],
    "Date": []
})

for Classifier_name, meth in zip(Classifier_names, meths):

    Rows = pd.DataFrame({
        "ROIs": [],
        "NROIs":[],
        "DataType": [],
        "freqs_band": [],  
        "subject": [],
        "Classifier": [],
        "Accuracy": [],
        "Features": [],
        "Comments": [],
        "Region Selection": [],
        "Date": []
    })
    print("\n\n"+"#"*30+"\n",DataType,Classifier_name)
    
    

    for subject_index in tqdm.tqdm(range(20)):
        tutti_fold_selezioni_regioni = []
        fold_accuracies = []
        all_predictions = []
        confusions_matrices = []
        
        data_2_Rest_EEG = read_subject(data_folder, "EEG", "Baseline",subject_index,verbose=verbose)
        data_2_MI_EEG = read_subject(data_folder, "EEG", "MI",subject_index,verbose=verbose)
        data_EEG = np.concatenate((data_2_Rest_EEG, data_2_MI_EEG), axis=0)
        WMI_EEG, WRe_EEG = compute_power_spectra(data_2_MI_EEG,data_2_Rest_EEG,starttime=start_time_selection_regions)
        data_EEG = freq_filter(data_EEG, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
        
        data_2_Rest_MEG = read_subject(data_folder, "MEG", "Baseline",subject_index,verbose=verbose)
        data_2_MI_MEG = read_subject(data_folder, "MEG", "MI",subject_index,verbose=verbose)
        data_MEG = np.concatenate((data_2_Rest_MEG, data_2_MI_MEG), axis=0)
        WMI_MEG, WRe_MEG = compute_power_spectra(data_2_MI_MEG,data_2_Rest_MEG, starttime=start_time_selection_regions)
        data_MEG = freq_filter(data_MEG, sf=250, freqs=[4,45], type_filter="bandpass") # axis = -1 by default
        

        # y shape: (n_trials,) with negative values for Rest and positive for MI
        y_EEG = np.concatenate([-np.ones((data_2_Rest_EEG.shape[0])), np.ones((data_2_MI_EEG.shape[0]))])
        y_MEG = np.concatenate([-np.ones((data_2_Rest_MEG.shape[0])), np.ones((data_2_MI_MEG.shape[0]))])

        assert np.array_equal(y_EEG, y_MEG), "Le etichette di EEG e MEG non corrispondono. Controlla i dati."


        # Data_EEG, WMI_EEG, labels_EEG, Feat_extraction_EEG, Feat_MI_EEG
       

        

        for fold_idx, (train_idx, block_idx) in enumerate(kf.split(data_EEG, y_EEG)):
            Feat_train_regions_selected = None
            Feat_test_regions_selected = None

            # selecting region based on training data only
            regions_selected_EEG = selezione_cohen(WRe_EEG, WMI_EEG, train_idx, num_best_ROIs)
            regions_selected_MEG = selezione_cohen(WRe_MEG, WMI_MEG, train_idx, num_best_ROIs)
            regions_selected = list(set(regions_selected_EEG) | set(regions_selected_MEG))
            
            regions_selected.sort()
            print("\n Per il fold ", fold_idx, " Ho selezionato un numero di regioni pari a ", len(regions_selected), regions_selected)
            tutti_fold_selezioni_regioni.append(regions_selected)


            for region_index, regione in enumerate(regions_selected):
        
                split_num = fold_idx+1
                
                x_train_MEG, y_train_MEG, x_val_MEG, y_val_MEG, x_test_MEG, y_test_MEG = (
                split_create_windows(data_MEG, y_MEG, train_idx, block_idx, 1, win_len=input_shape, shift=shift,
                    random_seed=random_seed, regione=regione,
                    start_train=start_train, end_train=end_train,
                    start_val_test=start_val_test, end_val_test=end_val_test)
                    )

                psd_train_MEG, psd_val_MEG, psd_test_MEG, freqs = compute_psd(
                x_train_MEG, x_val_MEG, x_test_MEG, y_train_MEG, y_val_MEG, y_test_MEG, 
                sampling_hz, fmin_spectra, fmax_spectra, bandwidth_spectra, log_spectra, scaler=None
                )
                
                x_train_EEG, y_train_EEG, x_val_EEG, y_val_EEG, x_test_EEG, y_test_EEG = (
                split_create_windows(data_EEG, y_EEG, train_idx, block_idx, 1, win_len=input_shape, shift=shift,
                    random_seed=random_seed, regione=regione,
                    start_train=start_train, end_train=end_train,
                    start_val_test=start_val_test, end_val_test=end_val_test)
                    )
                
                psd_train_EEG, psd_val_EEG, psd_test_EEG, freqs = compute_psd(
                x_train_EEG, x_val_EEG, x_test_EEG, y_train_EEG, y_val_EEG, y_test_EEG, 
                sampling_hz, fmin_spectra, fmax_spectra, bandwidth_spectra, log_spectra, scaler=None
                )
                
                assert np.array_equal(y_train_EEG, y_train_MEG), "Le etichette di train EEG e MEG non corrispondono. Controlla i dati."
                assert np.array_equal(y_val_EEG, y_val_MEG), "Le etichette di val EEG e MEG non corrispondono. Controlla i dati."
                assert np.array_equal(y_test_EEG, y_test_MEG), "Le etichette di test EEG e MEG non corrispondono. Controlla i dati."
            
        
                
                Features_training_1_regione_EEG = extract_features_from_spectra_more_bands_2(
                    psd_train_EEG, freqs,
                min_freq_frature=miff, max_freq_frature=maff, 
                select_mean=True, select_max=True,
                reshape_features=False)

                Features_test_1_regione_EEG = extract_features_from_spectra_more_bands_2(
                    psd_test_EEG, freqs,
                min_freq_frature=miff, max_freq_frature=maff, 
                select_mean=True, select_max=True,
                reshape_features=False)
            
                Features_training_1_regione_MEG = extract_features_from_spectra_more_bands_2(
                    psd_train_MEG, freqs,
                min_freq_frature=miff, max_freq_frature=maff, 
                select_mean=True, select_max=True,
                reshape_features=False)

                Features_test_1_regione_MEG = extract_features_from_spectra_more_bands_2(
                    psd_test_MEG, freqs,
                min_freq_frature=miff, max_freq_frature=maff, 
                select_mean=True, select_max=True,
                reshape_features=False)
                # Features_training_1_regione = model.encoder(x_train)
                # Features_test_1_regione = model.encoder(x_test)
                # Features_training_1_regione  shape (n_windows,16)
                Features_training_1_regione = np.concatenate((Features_training_1_regione_EEG, Features_training_1_regione_MEG), axis=-1)
                Features_test_1_regione = np.concatenate((Features_test_1_regione_EEG, Features_test_1_regione_MEG), axis=-1)

                
                    
                ################################### Feature Extractions ##########################################
                if Feat_train_regions_selected is None:
                    Feat_train_regions_selected = np.empty((Features_training_1_regione.shape[0],0))
                    Feat_test_regions_selected = np.empty((Features_test_1_regione.shape[0],0))

                Feat_train_regions_selected = np.concatenate((Feat_train_regions_selected,Features_training_1_regione), axis = 1) # asse di regione
                Feat_test_regions_selected = np.concatenate((Feat_test_regions_selected, Features_test_1_regione), axis = 1) # asse di regione

            #####################################################################################################################


            # if Classifier_name.lower()=="mlp":
            #     meth = build_compiled_mlp(input_dim=Feat_All_regions_selected_concat.shape[1])
                
            acc, cm, model_classifier, predictions = train_single_model_investigate2(Feat_train_regions_selected, Feat_test_regions_selected,
                                            (y_train_EEG > 0).astype(int), (y_test_EEG > 0).astype(int), return_predictions=True,
                                            method=meth, seed=224, fit_kwargs=MLP_fit_kwargs,
                                            n_windows = num_windows_val_test)
            

            fold_accuracies.append(acc)
            all_predictions.append(predictions)
            confusions_matrices.append(cm)


            if verbose:
                print(f"Fold {fold_idx+1} accuracy: {acc:.4f}, Confusion matrix:")
                print(cm)

            #####################################################################################################################

        new_row = {
                    "ROIs": [regions_selected],
                    "NROIs":[len(regions_selected)],
                    "DataType": [DataType],
                    "freqs_band": ["-"],  
                    "subject": [subject_index],
                    "Classifier": [Classifier_name], #SVC, #RandomForest, #LDA
                    "Accuracy": [fold_accuracies],
                    "Features": [Features_name],
                    "Comments": [Comments],
                    "Region Selection": [Region_Selection],
                    "Date": [today],
                    "Personalized_Freq_Range": ["-"],
                }
        Rows_all_subjects = pd.concat((Rows_all_subjects, pd.DataFrame(new_row)))

        # with open(f"../Results/BCI_Performances/Across_time_Power_Spectra_Confrontabili_with_DNN_windows_1s/Performances_across_time_subject_{subject_index}.pkl", "wb") as f:
        #     pickle.dump({
        #         "Details": new_row,
        #         "Selected_regions_per_fold_EEG_U_MEG": tutti_fold_selezioni_regioni,
        #         "all_predictions": all_predictions,
        #         "fold_accuracies": fold_accuracies
        #                 }, f)
Rows_all_subjects.to_pickle( "../Results/BCI_Performances/All_subjects_BCI_Performances_ultimate_region_selection_from_3s_windows_1s.pkl")



##############################
 EEG concat MEG SVC


  0%|          | 0/20 [00:00<?, ?it/s]


 Per il fold  0  Ho selezionato un numero di regioni pari a  13 [6, 15, 23, 27, 32, 33, 43, 45, 46, 48, 49, 51, 59]

 Per il fold  1  Ho selezionato un numero di regioni pari a  12 [15, 23, 27, 32, 33, 43, 44, 45, 46, 48, 49, 67]

 Per il fold  2  Ho selezionato un numero di regioni pari a  13 [15, 23, 27, 32, 33, 43, 44, 45, 46, 49, 51, 57, 59]

 Per il fold  3  Ho selezionato un numero di regioni pari a  13 [6, 15, 23, 27, 32, 43, 44, 45, 46, 48, 49, 51, 63]

 Per il fold  4  Ho selezionato un numero di regioni pari a  13 [6, 14, 15, 23, 27, 32, 33, 43, 44, 45, 48, 49, 59]


  5%|▌         | 1/20 [00:49<15:36, 49.31s/it]


 Per il fold  0  Ho selezionato un numero di regioni pari a  14 [6, 7, 14, 20, 23, 27, 32, 42, 43, 50, 51, 59, 61, 67]

 Per il fold  1  Ho selezionato un numero di regioni pari a  14 [1, 6, 7, 14, 23, 26, 31, 32, 43, 50, 51, 59, 61, 67]

 Per il fold  2  Ho selezionato un numero di regioni pari a  13 [1, 6, 7, 14, 23, 27, 32, 43, 50, 51, 59, 61, 67]

 Per il fold  3  Ho selezionato un numero di regioni pari a  13 [1, 6, 7, 14, 23, 26, 32, 43, 50, 51, 59, 61, 67]

 Per il fold  4  Ho selezionato un numero di regioni pari a  12 [6, 7, 14, 23, 26, 31, 43, 50, 51, 59, 61, 67]


...

 90%|█████████ | 18/20 [17:48<02:05, 62.57s/it]


 Per il fold  0  Ho selezionato un numero di regioni pari a  15 [2, 9, 18, 19, 25, 31, 33, 39, 44, 45, 47, 48, 49, 60, 61]

 Per il fold  1  Ho selezionato un numero di regioni pari a  16 [2, 3, 5, 9, 18, 25, 33, 39, 44, 45, 47, 48, 49, 60, 61, 67]

 Per il fold  2  Ho selezionato un numero di regioni pari a  14 [2, 3, 8, 18, 24, 33, 39, 44, 45, 48, 49, 52, 60, 61]

 Per il fold  3  Ho selezionato un numero di regioni pari a  16 [2, 3, 14, 18, 19, 33, 36, 38, 44, 45, 47, 48, 49, 60, 61, 64]

 Per il fold  4  Ho selezionato un numero di regioni pari a  15 [2, 3, 8, 18, 25, 33, 36, 39, 42, 44, 48, 60, 61, 65, 67]


 95%|█████████▌| 19/20 [18:49<01:02, 62.13s/it]


 Per il fold  0  Ho selezionato un numero di regioni pari a  13 [4, 32, 33, 44, 45, 46, 47, 48, 49, 59, 62, 63, 66]

 Per il fold  1  Ho selezionato un numero di regioni pari a  13 [4, 32, 33, 44, 45, 46, 47, 48, 50, 51, 59, 62, 63]

 Per il fold  2  Ho selezionato un numero di regioni pari a  12 [4, 32, 33, 44, 45, 46, 47, 48, 51, 59, 62, 63]

 Per il fold  3  Ho selezionato un numero di regioni pari a  13 [4, 32, 33, 44, 45, 46, 47, 48, 49, 51, 59, 62, 63]

 Per il fold  4  Ho selezionato un numero di regioni pari a  13 [4, 32, 33, 44, 45, 46, 47, 48, 51, 59, 62, 63, 66]


100%|██████████| 20/20 [19:47<00:00, 59.35s/it]


In [7]:
new_row

{'ROIs': [[[33, 32, 45, 49, 48, 44, 14, 59], [43, 27, 15, 23, 45, 49, 6, 44]]],
 'NROIs': [8],
 'DataType': ['EEG concat MEG'],
 'freqs_band': [((8, 12), (12, 30))],
 'subject': [0],
 'Classifier': ['SVC'],
 'Features': ['PowerSpectra - MeanMax band'],
 'Comments': ['Total regions = EEG_regions U MEG_regions'],
 'Region Selection': ["Cohen's d effect size selection starting from 3s"],
 'Date': [datetime.date(2026, 5, 25)],
 'Personalized_Freq_Range': [False]}